# Experiment 29: XGBoost Refinement

This experiment refines the strongest Experiment 23B approach.

The goal is to test a small number of controlled XGBoost variations while keeping
the proven leakage-safe exact-value target encoding and frequency encoding
pipeline unchanged.

Experiment 23B validation ROC-AUC: 0.945243


In [1]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

RANDOM_STATE = 42
N_SPLITS = 3
SMOOTHING = 20.0

DATA_DIR = Path("../data")
SUBMISSION_DIR = Path("../submissions")
RESULTS_DIR = Path("../results")

SUBMISSION_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)

train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")

TARGET = "Will_Buy_EV"

y = train[TARGET].map({"No": 0, "Yes": 1}).astype(np.int8)

X = train.drop(columns=[TARGET]).copy()
X_test = test.copy()

print("Train:", X.shape)
print("Test:", X_test.shape)
print("Positive rate:", round(y.mean(), 6))


Train: (668665, 14)
Test: (286571, 14)
Positive rate: 0.174645


In [2]:
# Base one-hot encoded features.

CATEGORICAL_COLS = X.select_dtypes(include=["object", "category"]).columns.tolist()

X_base = pd.get_dummies(
    X,
    columns=CATEGORICAL_COLS,
    dummy_na=True,
)

X_test_base = pd.get_dummies(
    X_test,
    columns=CATEGORICAL_COLS,
    dummy_na=True,
)

X_test_base = X_test_base.reindex(
    columns=X_base.columns,
    fill_value=0,
)

X_base = X_base.astype(np.float32)
X_test_base = X_test_base.astype(np.float32)

print("Base features:", X_base.shape)


Base features: (668665, 31)


In [3]:
# Exact-value identity columns used by Experiment 23B.

IDENTITY_COLS = [
    "Age",
    "Annual_Income_USD",
    "Daily_Commute_km",
    "Number_of_Cars_Owned",
    "Charging_Stations_Near_Home",
    "Charging_Stations_Near_Work",
    "Environmental_Concern_Level",
]


def add_exact_group_features(train_raw, train_y, apply_raw, columns, smoothing=20.0):
    train_extra = pd.DataFrame(index=apply_raw.index)

    global_mean = float(train_y.mean())

    for col in columns:
        stats = (
            pd.DataFrame({
                "key": train_raw[col].astype(str),
                "target": train_y.to_numpy(),
            })
            .groupby("key")["target"]
            .agg(["mean", "count"])
        )

        stats["encoded"] = (
            stats["mean"] * stats["count"] + global_mean * smoothing
        ) / (stats["count"] + smoothing)

        stats["frequency"] = stats["count"]

        keys = apply_raw[col].astype(str)

        train_extra[f"{col}_target_enc"] = (
            keys.map(stats["encoded"])
            .fillna(global_mean)
            .astype(np.float32)
            .to_numpy()
        )

        train_extra[f"{col}_freq"] = (
            keys.map(stats["frequency"])
            .fillna(0)
            .astype(np.float32)
            .to_numpy()
        )

    return train_extra


In [4]:
# Build leakage-safe OOF exact-value encodings.

cv = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE,
)

oof_extra = pd.DataFrame(
    np.nan,
    index=np.arange(len(X)),
    columns=[
        f"{col}_{feature}"
        for col in IDENTITY_COLS
        for feature in ["target_enc", "freq"]
    ],
)

test_extra_parts = []

for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y), start=1):
    fold_train = X.iloc[train_idx].reset_index(drop=True)
    fold_valid = X.iloc[valid_idx].reset_index(drop=True)
    fold_y = y.iloc[train_idx].reset_index(drop=True)

    valid_extra = add_exact_group_features(
        fold_train,
        fold_y,
        fold_valid,
        IDENTITY_COLS,
        smoothing=SMOOTHING,
    )

    oof_extra.iloc[valid_idx] = valid_extra.to_numpy()

    test_extra = add_exact_group_features(
        fold_train,
        fold_y,
        X_test.reset_index(drop=True),
        IDENTITY_COLS,
        smoothing=SMOOTHING,
    )

    test_extra_parts.append(test_extra)

oof_extra = oof_extra.fillna(float(y.mean()))

test_extra = (
    pd.concat(test_extra_parts, axis=0)
    .groupby(level=0)
    .mean()
    .reset_index(drop=True)
)

test_extra = test_extra.fillna(float(y.mean()))

X_features = pd.concat(
    [
        X_base.reset_index(drop=True),
        oof_extra.reset_index(drop=True),
    ],
    axis=1,
).astype(np.float32)

X_test_features = pd.concat(
    [
        X_test_base.reset_index(drop=True),
        test_extra.reset_index(drop=True),
    ],
    axis=1,
).astype(np.float32)

print("Final feature shape:", X_features.shape)


Final feature shape: (668665, 45)


In [5]:
# Controlled XGBoost variants around the Experiment 23B configuration.

MODEL_SPECS = {
    "XGB_29_01_23B": {
        "n_estimators": 1000,
        "max_depth": 5,
        "learning_rate": 0.035,
        "min_child_weight": 2,
        "subsample": 0.90,
        "colsample_bytree": 0.85,
        "reg_alpha": 0,
        "reg_lambda": 1,
    },
    "XGB_29_02_deeper": {
        "n_estimators": 1000,
        "max_depth": 6,
        "learning_rate": 0.035,
        "min_child_weight": 2,
        "subsample": 0.90,
        "colsample_bytree": 0.85,
        "reg_alpha": 0,
        "reg_lambda": 1,
    },
    "XGB_29_03_shallower": {
        "n_estimators": 1000,
        "max_depth": 4,
        "learning_rate": 0.035,
        "min_child_weight": 2,
        "subsample": 0.90,
        "colsample_bytree": 0.85,
        "reg_alpha": 0,
        "reg_lambda": 1,
    },
    "XGB_29_04_slow": {
        "n_estimators": 1400,
        "max_depth": 5,
        "learning_rate": 0.025,
        "min_child_weight": 2,
        "subsample": 0.90,
        "colsample_bytree": 0.85,
        "reg_alpha": 0,
        "reg_lambda": 1,
    },
    "XGB_29_05_fast": {
        "n_estimators": 700,
        "max_depth": 5,
        "learning_rate": 0.050,
        "min_child_weight": 2,
        "subsample": 0.90,
        "colsample_bytree": 0.85,
        "reg_alpha": 0,
        "reg_lambda": 1,
    },
    "XGB_29_06_regularized": {
        "n_estimators": 1000,
        "max_depth": 5,
        "learning_rate": 0.035,
        "min_child_weight": 3,
        "subsample": 0.90,
        "colsample_bytree": 0.85,
        "reg_alpha": 0.10,
        "reg_lambda": 2,
    },
    "XGB_29_07_high_subsample": {
        "n_estimators": 1000,
        "max_depth": 5,
        "learning_rate": 0.035,
        "min_child_weight": 2,
        "subsample": 0.95,
        "colsample_bytree": 0.90,
        "reg_alpha": 0,
        "reg_lambda": 1,
    },
    "XGB_29_08_low_child": {
        "n_estimators": 1000,
        "max_depth": 5,
        "learning_rate": 0.035,
        "min_child_weight": 1,
        "subsample": 0.90,
        "colsample_bytree": 0.85,
        "reg_alpha": 0,
        "reg_lambda": 1,
    },
}

print("Models:", len(MODEL_SPECS))


Models: 8


In [6]:
# Evaluate all variants with the same validation splits.

model_results = []
oof_predictions = {}

for model_name, params in MODEL_SPECS.items():
    print(f"\nTraining {model_name}...")

    oof_pred = np.zeros(len(X_features), dtype=np.float32)

    for fold, (train_idx, valid_idx) in enumerate(cv.split(X_features, y), start=1):
        model = XGBClassifier(
            **params,
            objective="binary:logistic",
            eval_metric="auc",
            tree_method="hist",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )

        model.fit(
            X_features.iloc[train_idx],
            y.iloc[train_idx],
        )

        oof_pred[valid_idx] = model.predict_proba(
            X_features.iloc[valid_idx]
        )[:, 1]

    score = roc_auc_score(y, oof_pred)

    model_results.append({
        "model": model_name,
        "oof_roc_auc": score,
    })

    oof_predictions[model_name] = oof_pred

    print(f"{model_name} | ROC-AUC: {score:.6f}")

model_results = (
    pd.DataFrame(model_results)
    .sort_values("oof_roc_auc", ascending=False)
    .reset_index(drop=True)
)

display(model_results)

model_results.to_csv(
    RESULTS_DIR / "experiment_29_model_results.csv",
    index=False,
)

pd.DataFrame(oof_predictions).to_csv(
    RESULTS_DIR / "experiment_29_oof_predictions.csv",
    index=False,
)

print("Saved Experiment 29 results.")



Training XGB_29_01_23B...
XGB_29_01_23B | ROC-AUC: 0.944802

Training XGB_29_02_deeper...
XGB_29_02_deeper | ROC-AUC: 0.944537

Training XGB_29_03_shallower...
XGB_29_03_shallower | ROC-AUC: 0.944848

Training XGB_29_04_slow...
XGB_29_04_slow | ROC-AUC: 0.944821

Training XGB_29_05_fast...
XGB_29_05_fast | ROC-AUC: 0.944737

Training XGB_29_06_regularized...
XGB_29_06_regularized | ROC-AUC: 0.944795

Training XGB_29_07_high_subsample...
XGB_29_07_high_subsample | ROC-AUC: 0.944855

Training XGB_29_08_low_child...
XGB_29_08_low_child | ROC-AUC: 0.944768


,model,oof_roc_auc
0,XGB_29_07_high_subsample,0.944855
1,XGB_29_03_shallower,0.944848
2,XGB_29_04_slow,0.944821
3,XGB_29_01_23B,0.944802
4,XGB_29_06_regularized,0.944795
5,XGB_29_08_low_child,0.944768
6,XGB_29_05_fast,0.944737
7,XGB_29_02_deeper,0.944537


Saved Experiment 29 results.


In [7]:
# Save the best model configuration for final retraining.

best_model_name = model_results.iloc[0]["model"]
best_score = model_results.iloc[0]["oof_roc_auc"]

print(f"Best model: {best_model_name}")
print(f"Best OOF ROC-AUC: {best_score:.6f}")

if best_score > 0.945243:
    print("NEW BEST: Experiment 29 beat Experiment 23B.")
else:
    print("Experiment 23B remains ahead.")


Best model: XGB_29_07_high_subsample
Best OOF ROC-AUC: 0.944855
Experiment 23B remains ahead.


## Experiment 29 checkpoint

The important comparison is against Experiment 23B's validation ROC-AUC of **0.945243**.

No submission is generated automatically here until the best variant is identified.
